# API Key

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
print(f'{os.environ['OPENAI_API_KEY'][:20]}...') 

sk-proj-iKU13YeoxNgF...


# import

In [2]:
import json

from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# from langchain_openai.chat_models import ChatOpenAI
# from langchain_openai.llms.base import OpenAI
# from langchain_core.prompts.prompt import PromptTemplate
# from langchain_core.prompts.chat import ChatPromptTemplate

# ↓ 걍 아래와 같이 해도 된다.
from langchain_openai import OpenAI, ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

from langchain.chat_models import init_chat_model

In [3]:
chat = ChatOpenAI(
    model="gpt-4o",
    temperature=0.1,
    streaming=True,
    callbacks=[
        StreamingStdOutCallbackHandler()
    ],
)

# PromptTemplate 포맷팅 방법2가지

In [4]:
# 방법1
t = PromptTemplate.from_template("What is the capital of {country}")
t.format(country="France")

'What is the capital of France'

In [5]:
# 방법2
t2 = PromptTemplate(
    template = "What is the capital of {country}",
    input_variables = ['country'],   # 입력변수들을 PromptTemplate 에 명시적으로 알려준다.
)

t2.format(country="Germany")


'What is the capital of Germany'

# FewShotPromptTemplate
모델에 예제(example) 추가

In [6]:
# 모델에게 '어떻게 대답해야 하는 지에 대한 예제(example)'를 AI 모델에게 주는 것이
# prompt 를 사용해서 '어떻게 대답해야 하는지 알려주는 것'보다 훨씬 좋다

# FewShotPromptTemplate 이 하는 일이 바로 그거다!
# - 이를 통해 예제(샘플)를 형식화(포맷) 할수 있다.
# - 이런 예제들을 데이터베이스등에 저장시켜놓고 활용할수도 있다

In [7]:
from langchain_core.prompts.few_shot import FewShotPromptTemplate

In [8]:
# 예제들.
# 모델이 나에게 '이런식으로 답변해 줬으면 좋겠다' 라고 제시하는 example 들.
examples = [
  {
    "question": "What do you know about France?",

    # ↓ 원하는 형식의 답변 이다..
    "answer": """
      Here is what I know:
      Capital: Paris
      Language: French
      Food: Wine and Cheese
      Currency: Euro
      """,
  },
  {
    "question": "What do you know about Italy?",
    "answer": """
      I know this:
      Capital: Rome
      Language: Italian
      Food: Pizza and Pasta
      Currency: Euro
      """,
  },
  {
    "question": "What do you know about Greece?",
    "answer": """
      I know this:
      Capital: Athens
      Language: Greek
      Food: Souvlaki and Feta Cheese
      Currency: Euro
      """,
  },    
]

In [9]:
# 일단 예제 없이 모델 호출
chat.invoke("What do you know about France?")

France, officially known as the French Republic, is a country located in Western Europe. It is known for its rich history, cultural heritage, and significant influence on global affairs. Here are some key points about France:

1. **Geography**: France is the largest country in the European Union by land area. It shares borders with Belgium, Luxembourg, Germany, Switzerland, Italy, Monaco, Spain, and Andorra. It also has coastlines along the Atlantic Ocean and the Mediterranean Sea.

2. **Capital and Major Cities**: The capital of France is Paris, which is also its largest city and a major global center for art, fashion, gastronomy, and culture. Other major cities include Marseille, Lyon, Toulouse, Nice, and Bordeaux.

3. **Language**: The official language is French, which is also one of the most widely spoken languages in the world.

4. **Government**: France is a semi-presidential representative democratic republic. The President is the head of state, while the Prime Minister is the 

AIMessage(content="France, officially known as the French Republic, is a country located in Western Europe. It is known for its rich history, cultural heritage, and significant influence on global affairs. Here are some key points about France:\n\n1. **Geography**: France is the largest country in the European Union by land area. It shares borders with Belgium, Luxembourg, Germany, Switzerland, Italy, Monaco, Spain, and Andorra. It also has coastlines along the Atlantic Ocean and the Mediterranean Sea.\n\n2. **Capital and Major Cities**: The capital of France is Paris, which is also its largest city and a major global center for art, fashion, gastronomy, and culture. Other major cities include Marseille, Lyon, Toulouse, Nice, and Bordeaux.\n\n3. **Language**: The official language is French, which is also one of the most widely spoken languages in the world.\n\n4. **Government**: France is a semi-presidential representative democratic republic. The President is the head of state, while

In [10]:
# 예제 사용

# 포맷 준비 <- examples 로 포맷팅 할거다
example_template = """
    Human: {question}
    AI: {answer}
"""
# ↑ {question} 과 {answer} 는 위 examples 의 샘플과 동일한 key 를 사용하여 작성.


In [11]:
example_prompt = PromptTemplate.from_template(example_template)

example_prompt

PromptTemplate(input_variables=['answer', 'question'], input_types={}, partial_variables={}, template='\n    Human: {question}\n    AI: {answer}\n')

In [12]:
# 데이터 한개로 포맷팅
print(example_prompt.format(**examples[2]))


    Human: What do you know about Greece?
    AI: 
      I know this:
      Capital: Athens
      Language: Greek
      Food: Souvlaki and Feta Cheese
      Currency: Euro
      



In [13]:
# 실제 prompt 작성
prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,   # 사용할 prompt
    examples = examples,   # 준비된 예제들

    # ↑ 이렇게만 해도 랭체인이 알아서 각각의 예제 리스트들을
    #  위 prompt 를 사용하여 포맷팅(형식화) 합니다  (밑)

    # 여기에 유저의 질문을 넣어줄겁니다

    # suffix=
    # 형식화된 모든 예제 마지막에 나오는 내용.
    # 그래서 한번 다 형식화가 끝난 다음, 사용자의 질문은 무엇인지 나옵니다

    # 여기서는 위에 있는 동일한 형식을 사용할 겁니다    

    suffix="Human: What do you know about {country}?",
    input_variables=['country']
)

prompt

FewShotPromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, examples=[{'question': 'What do you know about France?', 'answer': '\n      Here is what I know:\n      Capital: Paris\n      Language: French\n      Food: Wine and Cheese\n      Currency: Euro\n      '}, {'question': 'What do you know about Italy?', 'answer': '\n      I know this:\n      Capital: Rome\n      Language: Italian\n      Food: Pizza and Pasta\n      Currency: Euro\n      '}, {'question': 'What do you know about Greece?', 'answer': '\n      I know this:\n      Capital: Athens\n      Language: Greek\n      Food: Souvlaki and Feta Cheese\n      Currency: Euro\n      '}], example_prompt=PromptTemplate(input_variables=['answer', 'question'], input_types={}, partial_variables={}, template='\n    Human: {question}\n    AI: {answer}\n'), suffix='Human: What do you know about {country}?')

In [14]:
print(prompt.format(country="Germany"))


    Human: What do you know about France?
    AI: 
      Here is what I know:
      Capital: Paris
      Language: French
      Food: Wine and Cheese
      Currency: Euro
      



    Human: What do you know about Italy?
    AI: 
      I know this:
      Capital: Rome
      Language: Italian
      Food: Pizza and Pasta
      Currency: Euro
      



    Human: What do you know about Greece?
    AI: 
      I know this:
      Capital: Athens
      Language: Greek
      Food: Souvlaki and Feta Cheese
      Currency: Euro
      


Human: What do you know about Germany?


In [15]:
# step1  example 리스트를 만들고   examples
# step2  FewShotPromptTemplate 에 전달했고 examples=
# step3  어떻게 전달한 예제들을 형식화 할지 알려주었고
# step4  마지막에 질문을 포함시켰다.  suffix, input_variables

# AI 는 우리의 예제들과 똑같은 구조, 형태로 답변하게 될겁니다


In [16]:
chain = prompt | chat

chain

FewShotPromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, examples=[{'question': 'What do you know about France?', 'answer': '\n      Here is what I know:\n      Capital: Paris\n      Language: French\n      Food: Wine and Cheese\n      Currency: Euro\n      '}, {'question': 'What do you know about Italy?', 'answer': '\n      I know this:\n      Capital: Rome\n      Language: Italian\n      Food: Pizza and Pasta\n      Currency: Euro\n      '}, {'question': 'What do you know about Greece?', 'answer': '\n      I know this:\n      Capital: Athens\n      Language: Greek\n      Food: Souvlaki and Feta Cheese\n      Currency: Euro\n      '}], example_prompt=PromptTemplate(input_variables=['answer', 'question'], input_types={}, partial_variables={}, template='\n    Human: {question}\n    AI: {answer}\n'), suffix='Human: What do you know about {country}?')
| ChatOpenAI(callbacks=[<langchain_core.callbacks.streaming_stdout.StreamingStdOutCallbackHandler object at

In [17]:
chain.invoke({"country": "Germany"})

AI: 
  Here's what I know:
  Capital: Berlin
  Language: German
  Food: Bratwurst and Sauerkraut
  Currency: Euro
  

AIMessage(content="AI: \n  Here's what I know:\n  Capital: Berlin\n  Language: German\n  Food: Bratwurst and Sauerkraut\n  Currency: Euro\n  ", additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_7c2c459cd0', 'service_tier': 'default', 'model_provider': 'openai'}, id='lc_run--01a18c15-2606-7ad1-bbde-8dd8b3cc81c5', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 153, 'output_tokens': 37, 'total_tokens': 190, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [18]:
chain.invoke({"country": "Turkey"})

AI: 
  Here is what I know:
  Capital: Ankara
  Language: Turkish
  Food: Kebab and Baklava
  Currency: Turkish Lira
  

AIMessage(content='AI: \n  Here is what I know:\n  Capital: Ankara\n  Language: Turkish\n  Food: Kebab and Baklava\n  Currency: Turkish Lira\n  ', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_7c2c459cd0', 'service_tier': 'default', 'model_provider': 'openai'}, id='lc_run--01a18c15-2c8b-7411-855d-d52b8cbe2812', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 153, 'output_tokens': 37, 'total_tokens': 190, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

# FewShotChatMessagePromptTemplate

In [19]:
from langchain_core.prompts.few_shot import FewShotChatMessagePromptTemplate

In [20]:
examples = [
  {
    "country": "France",

    "answer": """
      Here is what I know:
      Capital: Paris
      Language: French
      Food: Wine and Cheese
      Currency: Euro
      """,
  },
  {
    "country": "Italy",
    "answer": """
      I know this:
      Capital: Rome
      Language: Italian
      Food: Pizza and Pasta
      Currency: Euro
      """,
  },
  {
    "country": "Greece",
    "answer": """
      I know this:
      Capital: Athens
      Language: Greek
      Food: Souvlaki and Feta Cheese
      Currency: Euro
      """,
  },    
]

In [21]:
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "What do you know abount {country}?"),  # <- 위 examples 와 동일한 key 로!
    ("ai", "{answer}"),
])

example_prompt = FewShotChatMessagePromptTemplate(
    example_prompt = example_prompt,
    examples=examples,
    # suffix= 등은 필요없다.
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a geography expert, you give short answers."),
    example_prompt, # ★
    ("human", "What do you know about {country}?"),
])

final_prompt.format_messages(country="Germany")

[SystemMessage(content='You are a geography expert, you give short answers.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What do you know abount France?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='\n      Here is what I know:\n      Capital: Paris\n      Language: French\n      Food: Wine and Cheese\n      Currency: Euro\n      ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What do you know abount Italy?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='\n      I know this:\n      Capital: Rome\n      Language: Italian\n      Food: Pizza and Pasta\n      Currency: Euro\n      ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What do you know abount Greece?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='\n      I know this:\n      Capital: Athens\n      Language: Greek\n      Food: So

In [22]:
chain = final_prompt | chat

chain.invoke({"country": "Germany"})

I know this:  
Capital: Berlin  
Language: German  
Food: Bratwurst and Sauerkraut  
Currency: Euro  

AIMessage(content='I know this:  \nCapital: Berlin  \nLanguage: German  \nFood: Bratwurst and Sauerkraut  \nCurrency: Euro  ', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_03eab4df58', 'service_tier': 'default', 'model_provider': 'openai'}, id='lc_run--01a18c15-321a-70c1-9bbf-a7a038f14a4b', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 171, 'output_tokens': 28, 'total_tokens': 199, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [23]:
chain.invoke({"country": "South Korea"})

I know this:  
Capital: Seoul  
Language: Korean  
Food: Kimchi and Bibimbap  
Currency: South Korean Won  

AIMessage(content='I know this:  \nCapital: Seoul  \nLanguage: Korean  \nFood: Kimchi and Bibimbap  \nCurrency: South Korean Won  ', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_03eab4df58', 'service_tier': 'default', 'model_provider': 'openai'}, id='lc_run--01a18c15-36c0-79f1-978c-f8b0eea636a5', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 172, 'output_tokens': 28, 'total_tokens': 200, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [24]:
#  다음에는 '예제' 를 어떻게 선택하는지에 대해 배워보자.

#  때로는 수천개의 예제를 가지고 있을텐데,  이를 모두 모델에게 줄수 없는 상황이 있을수 있다.
#    이유1) 비용이 많이 든다..  많은 텍스트 땜에.
#    이유2) '허용하는 범위' 라는게 있다 => 모~든 예제들을 모델에게 줄 수는 없다.  
#           제한이 있다 (context window)

#  그래서 예제를 선별하는 방법에 대해 배워보자

# ExampleSelector

## LengthBasedExampleSelector

In [25]:
from langchain_core.example_selectors.length_based import LengthBasedExampleSelector

In [26]:
# LengthBasedExampleSelector 는 기본적으로
# - 예제(example) 들을 형식화 할 수 있고
# - 예제의 양이 얼마나 되는지를 확인할수 있다.

# 그러면, 사용자가 설정해 놓은 세팅값에 따라 prompt 에 알맞은 예제를 골라준다.

In [27]:
example_prompt = PromptTemplate.from_template("Human: {country}\nAI: {answer}")


example_prompt

PromptTemplate(input_variables=['answer', 'country'], input_types={}, partial_variables={}, template='Human: {country}\nAI: {answer}')

In [28]:
example_selector = LengthBasedExampleSelector(
    examples=examples, 
    example_prompt=example_prompt,  # 포맷팅 의 분량이 얼마나 되는지 알아보기 위해 example_prompt 필요.
    max_length=10,  # 예제의 양을 얼마나 허용할지 정해주기, max_length= 값 위(?)의 예제는 cut out  된다.
          # ↑ 예제에서 함 10으로 지정해본다.
                #  LengthBasedExampleSelector 의 max_length 는 '단어의 개수' 단위
                #  '글자의 개수' 나 '토큰의 개수'가 아니다    
)

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    # examples= 대신에 example_selector= 지정
    example_selector=example_selector,   #  <- max_length 값에 따라 예제의 양을 정해줄 수 있다.

    suffix="Human: What do you know about {country}?",
    input_variables=['country']   
)

prompt.format(country='Brazil')  # 확인


'Human: What do you know about Brazil?'

In [29]:
# max_length= 10 .  <- 너무 작아서 example 한개도 포맷팅 되지 않음!

In [30]:
example_selector = LengthBasedExampleSelector(
    examples=examples, 
    example_prompt=example_prompt,
    max_length=80,  # <- 변경!
)

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,
    suffix="Human: What do you know about {country}?",
    input_variables=['country']   
)

print(prompt.format(country='Brazil'))  # 확인


Human: France
AI: 
      Here is what I know:
      Capital: Paris
      Language: French
      Food: Wine and Cheese
      Currency: Euro
      

Human: What do you know about Brazil?


In [31]:
example_selector = LengthBasedExampleSelector(
    examples=examples, 
    example_prompt=example_prompt,
    max_length=180,  # <- 변경!
)

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,
    suffix="Human: What do you know about {country}?",
    input_variables=['country']   
)

print(prompt.format(country='Brazil'))  # 확인


Human: France
AI: 
      Here is what I know:
      Capital: Paris
      Language: French
      Food: Wine and Cheese
      Currency: Euro
      

Human: Italy
AI: 
      I know this:
      Capital: Rome
      Language: Italian
      Food: Pizza and Pasta
      Currency: Euro
      

Human: Greece
AI: 
      I know this:
      Capital: Athens
      Language: Greek
      Food: Souvlaki and Feta Cheese
      Currency: Euro
      

Human: What do you know about Brazil?


## Custom ExampleSelector (BaseExampleSelector)

In [32]:
# 내가 원하는 대로의 example 들을 허용할수 있도록 ExampleSelector 제공 가능.

In [33]:
from langchain_core.example_selectors.base import BaseExampleSelector

In [34]:
# BaseExampleSelector 의 구현객체를 만드려면
#  상속 받은뒤 select_examples() 과 add_example() 을 반드시 오버라이딩 해주어야 한다.

class RandomExampleSelector(BaseExampleSelector):
    def __init__(self, examples):
        self.examples = examples


    def select_examples(self, input_variables):
        """입력에 따라 어떠한 샘플을 사용할지 select 함"""
        from random import choice
        return [choice(self.examples)]  

    def add_example(self, example):
        """이미 존재하는 example 에 example 을 추가"""
        self.examples.append(example)






In [35]:
example_selector = RandomExampleSelector(examples=examples)

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,
    suffix="Human: What do you know about {country}?",
    input_variables=['country']   
)

print(prompt.format(country='Brazil'))  # 확인

Human: France
AI: 
      Here is what I know:
      Capital: Paris
      Language: French
      Food: Wine and Cheese
      Currency: Euro
      

Human: What do you know about Brazil?


# PromptTemplate 저장/읽어오기
- prompt 를 `JSON` 으로 저장(직렬화)  불러오기 (역직렬화) 가능

`langchain_core.load`의 네 함수는 LangChain 객체(프롬프트, 메시지, 체인 등)를 **직렬화/역직렬화**하는 도구입니다.

**함수 역할**

| 함수 | 역할 |
|---|---|
| `dumpd(obj)` | 객체 → **dict** (JSON 호환 파이썬 딕셔너리) |
| `dumps(obj)` | 객체 → **JSON 문자열** |
| `load(dict)` | dict → 객체 복원 |
| `loads(str)` | JSON 문자열 → 객체 복원 |

이름 규칙은 json 모듈과 같습니다. `s`가 붙으면 문자열(string)을 다룹니다.



In [36]:
from langchain_core.load import dumpd, dumps, load, loads

In [37]:
# prompt 를 파일로 다룰 경우
#   'JSON' 파일  혹은 'YAML' 파일 로 만들수 있다

"""
디스크에서 prompt templates 를 가져오는 방법에 대해 배워볼겁니다.

가령, 회사에서
프롬프트 엔지니어링을 하는 사람이 있고,
당신이 '개발자' 이며,  해당 프롬프트를 가져와야 하는 상황이라던가
어딘가 저장해두고, 다른 누구나 프롬프트를 가져다 쓸수 있도록 하고 싶다면
그런 상황에서 유용합니다

이를 위해서, 먼저 두가지 타입의 prompt 를 만들어야 합니다
하나는 Json 파일이고, 다른하나는 yaml 파일입니다
"""
None

## json 파일

In [38]:
with open("prompt.json", "w") as f:
    f.write("""
    {
        "_type": "prompt",
        "template": "What is the capital of {country}",
        "input_variables": ["country"]
    }
    """)

In [39]:
!type prompt.json


    {
        "_type": "prompt",
        "template": "What is the capital of {country}",
        "input_variables": ["country"]
    }
    


In [40]:
with open("prompt.json", encoding="utf-8") as f:
    prompt = loads(f.read())

type(prompt)

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_3056\1839210122.py:2: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  prompt = loads(f.read())
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_3056\1839210122.py:2: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  prompt = loads(f.read())


dict

In [41]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role} 전문가입니다."),
    ("human", "{question}"),    
])

In [42]:
d = dumpd(prompt)   # dict
d

{'lc': 1,
 'type': 'constructor',
 'id': ['langchain', 'prompts', 'chat', 'ChatPromptTemplate'],
 'kwargs': {'input_variables': ['question', 'role'],
  'messages': [{'lc': 1,
    'type': 'constructor',
    'id': ['langchain', 'prompts', 'chat', 'SystemMessagePromptTemplate'],
    'kwargs': {'prompt': {'lc': 1,
      'type': 'constructor',
      'id': ['langchain', 'prompts', 'prompt', 'PromptTemplate'],
      'kwargs': {'input_variables': ['role'],
       'template': '당신은 {role} 전문가입니다.',
       'template_format': 'f-string'},
      'name': 'PromptTemplate'}}},
   {'lc': 1,
    'type': 'constructor',
    'id': ['langchain', 'prompts', 'chat', 'HumanMessagePromptTemplate'],
    'kwargs': {'prompt': {'lc': 1,
      'type': 'constructor',
      'id': ['langchain', 'prompts', 'prompt', 'PromptTemplate'],
      'kwargs': {'input_variables': ['question'],
       'template': '{question}',
       'template_format': 'f-string'},
      'name': 'PromptTemplate'}}}]},
 'name': 'ChatPromptTemplate'

In [43]:
s = dumps(prompt) # JSON
print(s)

{"lc": 1, "type": "constructor", "id": ["langchain", "prompts", "chat", "ChatPromptTemplate"], "kwargs": {"input_variables": ["question", "role"], "messages": [{"lc": 1, "type": "constructor", "id": ["langchain", "prompts", "chat", "SystemMessagePromptTemplate"], "kwargs": {"prompt": {"lc": 1, "type": "constructor", "id": ["langchain", "prompts", "prompt", "PromptTemplate"], "kwargs": {"input_variables": ["role"], "template": "\ub2f9\uc2e0\uc740 {role} \uc804\ubb38\uac00\uc785\ub2c8\ub2e4.", "template_format": "f-string"}, "name": "PromptTemplate"}}}, {"lc": 1, "type": "constructor", "id": ["langchain", "prompts", "chat", "HumanMessagePromptTemplate"], "kwargs": {"prompt": {"lc": 1, "type": "constructor", "id": ["langchain", "prompts", "prompt", "PromptTemplate"], "kwargs": {"input_variables": ["question"], "template": "{question}", "template_format": "f-string"}, "name": "PromptTemplate"}}}]}, "name": "ChatPromptTemplate"}


In [44]:
with open("prompt.json", "w", encoding='utf-8') as f:
    f.write(dumps(prompt, pretty=True))

In [45]:
# 불러오기 ()
prompt2 = loads(s)  # 문자열
prompt2

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_3056\1760167368.py:2: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  prompt2 = loads(s)  # 문자열


ChatPromptTemplate(input_variables=['question', 'role'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['role'], input_types={}, partial_variables={}, template='당신은 {role} 전문가입니다.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})])

In [46]:
prompt3 = load(d)
prompt3

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_3056\720372209.py:1: LangChainBetaWarning: The function `load` is in beta. It is actively being worked on, so the API may change.
  prompt3 = load(d)
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_3056\720372209.py:1: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  prompt3 = load(d)


ChatPromptTemplate(input_variables=['question', 'role'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['role'], input_types={}, partial_variables={}, template='당신은 {role} 전문가입니다.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})])

In [47]:
with open("prompt.json", encoding='utf-8') as f:
    prompt4 = loads(f.read())

prompt4

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_3056\2489929777.py:2: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  prompt4 = loads(f.read())


ChatPromptTemplate(input_variables=['question', 'role'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['role'], input_types={}, partial_variables={}, template='당신은 {role} 전문가입니다.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})])

In [48]:
prompt4.format_messages(question='변수가 뭐야', role='파이썬')

[SystemMessage(content='당신은 파이썬 전문가입니다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='변수가 뭐야', additional_kwargs={}, response_metadata={})]

In [49]:
prompt4.invoke({"question":'변수가 뭐야', "role":'파이썬'})

ChatPromptValue(messages=[SystemMessage(content='당신은 파이썬 전문가입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='변수가 뭐야', additional_kwargs={}, response_metadata={})])

# Caching

In [ ]:
# Caching 을 사용하면 모델의 응답을 저장(cache)할수 있다.
# 예를들어
# 똑같은 질문을 받는 상황이라면 그 때마다 답변생성할 필요 없이
# 이미 캐싱된 답변을 재사용 할수 있는 것이다 -->  비용절감! 

## set_llm_cache(), InMemoryCache

In [50]:
from langchain_core.globals import set_llm_cache
from langchain_core.caches import InMemoryCache

In [51]:
# ↓ 이와 같이 세팅하면 LLM 의 모든 response 가 '메모리' 에 저장된다.
set_llm_cache(InMemoryCache())

In [52]:
chat = ChatOpenAI(
    model='gpt-4o',
    temperature=0.1,
)

In [53]:
import time
from datetime import timedelta

def check_laptime(message):
    start_time = time.time()
    response = chat.invoke(message)
    end_time = time.time()
    elapsed_time = end_time - start_time # 경과시간
    print('▶ 경과시간 %s' % (str(timedelta(seconds = elapsed_time))))
    print(f'{len(response.content)} 글자: {response.content}\n')


In [54]:
check_laptime("How do you make italian pasta")

▶ 경과시간 0:00:06.475946
2694 글자: Making Italian pasta from scratch is a rewarding process that involves just a few simple ingredients. Here's a basic recipe to guide you through making fresh pasta dough:

### Ingredients:
- 2 cups (about 250g) all-purpose flour or "00" flour (for a finer texture)
- 3 large eggs
- A pinch of salt
- Semolina flour or additional all-purpose flour for dusting

### Instructions:

1. **Prepare the Workspace:**
   - Clean a large, flat surface (like a countertop or large cutting board) for mixing and kneading the dough.

2. **Make the Dough:**
   - Place the flour on the surface and create a well in the center.
   - Crack the eggs into the well and add a pinch of salt.
   - Using a fork, beat the eggs gently, gradually incorporating the flour starting from the inner rim of the well. Continue until the mixture is too stiff to mix with a fork.

3. **Knead the Dough:**
   - Bring the dough together with your hands and begin kneading it. Knead for about 8-10 minute

In [55]:
# 동일한 프롬프트로 두번째 호출시, 
# 첫번째 호출로 cache 된 결과 리턴.  (llm 호출 발생 안함.)
check_laptime("How do you make italian pasta")

▶ 경과시간 0:00:00.000529
2694 글자: Making Italian pasta from scratch is a rewarding process that involves just a few simple ingredients. Here's a basic recipe to guide you through making fresh pasta dough:

### Ingredients:
- 2 cups (about 250g) all-purpose flour or "00" flour (for a finer texture)
- 3 large eggs
- A pinch of salt
- Semolina flour or additional all-purpose flour for dusting

### Instructions:

1. **Prepare the Workspace:**
   - Clean a large, flat surface (like a countertop or large cutting board) for mixing and kneading the dough.

2. **Make the Dough:**
   - Place the flour on the surface and create a well in the center.
   - Crack the eggs into the well and add a pinch of salt.
   - Using a fork, beat the eggs gently, gradually incorporating the flour starting from the inner rim of the well. Continue until the mixture is too stiff to mix with a fork.

3. **Knead the Dough:**
   - Bring the dough together with your hands and begin kneading it. Knead for about 8-10 minute

## set_debug()

In [56]:
from langchain_core.globals import set_debug

In [57]:
set_debug(True)

In [58]:
chat.invoke("How do you make italian pasta")

[llm/start] [llm:ChatOpenAI] Entering LLM run with input:
{
  "prompts": [
    "Human: How do you make italian pasta"
  ]
}
[llm/end] [llm:ChatOpenAI] s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "Making Italian pasta from scratch is a rewarding process that involves just a few simple ingredients. Here's a basic recipe to guide you through making fresh pasta dough:\n\n### Ingredients:\n- 2 cups (about 250g) all-purpose flour or \"00\" flour (for a finer texture)\n- 3 large eggs\n- A pinch of salt\n- Semolina flour or additional all-purpose flour for dusting\n\n### Instructions:\n\n1. **Prepare the Workspace:**\n   - Clean a large, flat surface (like a countertop or large cutting board) for mixing and kneading the dough.\n\n2. **Make the Dough:**\n   - Place the flour on the surface and create a well in the center.\n   - Crack the eggs into the well and add a pinch of salt.\n   - Using a fork, beat the eggs gently, gradually incorporating the flour 

AIMessage(content='Making Italian pasta from scratch is a rewarding process that involves just a few simple ingredients. Here\'s a basic recipe to guide you through making fresh pasta dough:\n\n### Ingredients:\n- 2 cups (about 250g) all-purpose flour or "00" flour (for a finer texture)\n- 3 large eggs\n- A pinch of salt\n- Semolina flour or additional all-purpose flour for dusting\n\n### Instructions:\n\n1. **Prepare the Workspace:**\n   - Clean a large, flat surface (like a countertop or large cutting board) for mixing and kneading the dough.\n\n2. **Make the Dough:**\n   - Place the flour on the surface and create a well in the center.\n   - Crack the eggs into the well and add a pinch of salt.\n   - Using a fork, beat the eggs gently, gradually incorporating the flour starting from the inner rim of the well. Continue until the mixture is too stiff to mix with a fork.\n\n3. **Knead the Dough:**\n   - Bring the dough together with your hands and begin kneading it. Knead for about 8-1

## debug, cache 끄기

In [59]:
set_debug(False)

In [60]:
set_llm_cache(None)

# Usage 사용량 체크

In [61]:
# 모델 호출 사용량 확인
from langchain_core.callbacks import get_usage_metadata_callback

In [62]:
with get_usage_metadata_callback() as cb:
    chat.invoke("Hello")
    chat.invoke("Hi again")
    print(cb.usage_metadata)

{'gpt-4o-2024-08-06': {'output_tokens': 18, 'total_tokens': 35, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'input_tokens': 17, 'output_token_details': {'audio': 0, 'reasoning': 0}}}


In [64]:
from langchain_core.callbacks import UsageMetadataCallbackHandler

callback = UsageMetadataCallbackHandler()

chat.invoke("Hello", config={"callbacks": [callback]})

print(callback.usage_metadata)


{'gpt-4o-2024-08-06': {'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}}


In [65]:
PRICE = {"gpt-4o": (2.5, 10)}   # USD  / 1M tokens (입력, 출력)

In [67]:
for model, u in cb.usage_metadata.items():
    key = next(k for k in PRICE if model.startswith(k))
    inp, out = PRICE[key]
    cost = (u['input_tokens'] * inp + u['output_tokens'] * out) / 1_000_000
    print(model, f"${cost:.6f}")
    

gpt-4o-2024-08-06 $0.000222


# 모델 저장 / 불러오기

## 방법1 : LangChain 공식 직렬화 (dumps / loads)

In [72]:
chat = ChatOpenAI(
    model="gpt-5.5",
    temperature=0.1,
    max_tokens=450,
)

# 모델 저장
with open('model.json', 'w', encoding='utf-8') as f:
    f.write(dumps(chat, pretty=True))

In [73]:
!type model.json

{
  "lc": 1,
  "type": "constructor",
  "id": [
    "langchain",
    "chat_models",
    "openai",
    "ChatOpenAI"
  ],
  "kwargs": {
    "model_name": "gpt-5.5",
    "openai_api_key": {
      "lc": 1,
      "type": "secret",
      "id": [
        "OPENAI_API_KEY"
      ]
    },
    "stream_usage": true,
    "max_tokens": 450
  },
  "name": "ChatOpenAI"
}


In [74]:
# 불러오기
with open('model.json', encoding='utf-8') as f:
    chat = loads(
        f.read(),
        allowed_objects=[ChatOpenAI],  # 복원을 허용할 클래스 지정
        secrets_from_env=True,    # OPENAI_API_KEY 를 환경변수에서 읽음.
    )

In [75]:
print(chat.invoke("안녕?").content)

안녕하세요! 무엇을 도와드릴까요?


📌주의할 점:

- API 키는 JSON에 실제 값이 아니라 `OPENAI_API_KEY`라는 참조로만 저장됩니다. 그래서 불러올 때 키를 넘겨줘야 합니다.

- 최근 보안 패치로 기본값이 바뀌었습니다. `load()`/`loads()`에 `allowed_objects="core"` 허용 목록이 기본 적용되고, `secrets_from_env`는 이제 기본값이 False입니다. 기본 허용 목록은 메시지·프롬프트 같은 core 타입만 포함하므로, `ChatOpenAI`를 복원하려면 위처럼 `allowed_objects`를 직접 지정해야 합니다. 환경변수 대신 `secrets_map={"OPENAI_API_KEY": "sk-..."}`로 키를 넘길 수도 있습니다. 

- 신뢰할 수 없거나 인증되지 않은 출처의 데이터에는 절대 `load()`를 호출하지 말라는 경고가 있으니, 주고받은 JSON을 그대로 로드하는 상황은 피하는 게 좋습니다

## 방법2 : 설정값만 JSON 으로 저장

In [80]:
config = {
    "model": "gpt-5.5",
    "model_provider": "openai",
    "temperature": 0.1,
    "max_tokens": 450,
}

with open("model.json", "w", encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

In [81]:
!type model.json

{
  "model": "gpt-5.5",
  "model_provider": "openai",
  "temperature": 0.1,
  "max_tokens": 450
}


In [82]:
with open("model.json", encoding='utf-8') as f:
    chat = init_chat_model(**json.load(f))

In [83]:
chat.invoke('hello').content

'Hello! How can I help you today?'